In [ ]:
import os
import urllib.request
import time
import akshare as ak
import pandas as pd

# ==========================================
# 终极网络环境清理（屏蔽所有系统代理）
# ==========================================
os.environ["http_proxy"] = ""
os.environ["https_proxy"] = ""
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["all_proxy"] = ""
os.environ["ALL_PROXY"] = ""
# 暴力覆盖底层获取代理的函数，让 requests 彻底变成"瞎子"，只能直连
urllib.request.getproxies = lambda: {}

def get_target_stocks_akshare():
    print("正在通过 AkShare 获取全市场 A 股最新实时/盘后数据...")

    # ==========================================
    # 增加容错与重试机制
    # ==========================================
    max_retries = 3
    df = pd.DataFrame()
    for attempt in range(max_retries):
        try:
            # 使用 stock_zh_a_spot_em() 获取东方财富实时行情（包含市盈率、市净率、总市值等估值字段）
            # 注意：旧版 stock_zh_a_spot() 在 akshare 1.18+ 已不再返回估值列
            df = ak.stock_zh_a_spot_em()
            print(f"✅ 全市场数据拉取成功！共 {len(df)} 只股票")
            break
        except Exception as e:
            print(f"⚠️ 第 {attempt + 1} 次尝试拉取失败 (原因: {e})。")
            if attempt < max_retries - 1:
                print("⏳ 等待 3 秒后重试...")
                time.sleep(3)
            else:
                print("❌ 已达到最大重试次数，请稍后再试或检查网络状态。")
                print("提示：如果持续报 ProxyError/ConnectionError，请检查系统代理设置或 VPN。")
                return pd.DataFrame()

    if df.empty:
        return df

    print(f"共获取到 {len(df)} 只股票，开始进行清洗和过滤...")

    # 检查必要的列是否存在（兼容不同 akshare 版本的列名差异）
    # stock_zh_a_spot_em() 返回的列名示例：
    # 代码, 名称, 最新价, 涨跌幅, 涨跌额, 成交量, 成交额, 振幅, 最高, 最低, 今开, 昨收,
    # 量比, 换手率, 市盈率-动态, 市净率, 总市值, 流通市值
    required_cols = ['市盈率-动态', '市净率', '总市值']
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        print(f"⚠️ 当前接口未返回以下列: {missing_cols}")
        print(f"实际返回的列: {list(df.columns)}")
        print("尝试使用备用接口 stock_zh_a_spot()...")
        # 备用方案：stock_zh_a_spot() 返回的是新浪行情，列较少但可获取基础数据
        try:
            df = ak.stock_zh_a_spot()
            print(f"备用接口返回列: {list(df.columns)}")
        except Exception as e2:
            print(f"备用接口也失败: {e2}")
            return pd.DataFrame()
        # 如果备用接口也没有估值列，则无法继续
        still_missing = [c for c in required_cols if c not in df.columns]
        if still_missing:
            print(f"❌ 备用接口也缺少估值列: {still_missing}")
            print("建议：升级 akshare 到最新版本 pip install --upgrade akshare")
            return pd.DataFrame()

    # 1. 过滤掉停牌股票（成交量为0或最新价为0）
    df = df[df['成交量'] > 0].copy()
    df = df.dropna(subset=['最新价', '市盈率-动态', '市净率', '总市值'])

    # 确保数值列为 float 类型
    for col in ['最新价', '市盈率-动态', '市净率', '总市值', '流通市值']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['最新价', '市盈率-动态', '市净率', '总市值'])

    # 2. 过滤非 ST 股票
    df = df[~df['名称'].str.contains('ST', na=False)]

    # 3. 过滤市盈率 > 0 且 市净率 > 0
    df = df[(df['市盈率-动态'] > 0) & (df['市净率'] > 0)]

    # 4. 过滤沪深主板股票（仅保留 60 和 00 开头的股票）
    main_board_mask = df['代码'].str.contains(r'^(60|00)', regex=True)
    df = df[main_board_mask]

    if df.empty:
        print("没有符合条件的股票。")
        return df

    # 5. 按市值从小到大排序，并截取前 50 只
    top_50_smallest_cap = df.sort_values(by='总市值', ascending=True).head(50)

    # 6. 在这 50 只股票的基础上，按价格由低到高排序
    final_result = top_50_smallest_cap.sort_values(by='最新价', ascending=True)

    # 重置索引并格式化输出
    final_result = final_result.reset_index(drop=True)

    # 将市值转换为"亿元"单位，保留两位小数，方便阅读
    final_result = final_result.copy()  # 避免 SettingWithCopyWarning
    final_result['总市值(亿)'] = (final_result['总市值'] / 100000000).round(2)
    if '流通市值' in final_result.columns:
        final_result['流通市值(亿)'] = (final_result['流通市值'] / 100000000).round(2)

    # 选取需要展示的核心列
    columns_to_show = ['代码', '名称', '最新价', '市盈率-动态', '市净率', '总市值(亿)']
    if '流通市值(亿)' in final_result.columns:
        columns_to_show.append('流通市值(亿)')

    return final_result[columns_to_show]

if __name__ == "__main__":
    result_df = get_target_stocks_akshare()

    if not result_df.empty:
        print("\n=== 最终筛选结果（主板中市值最小的前50只，并按最新价由低到高排序） ===")
        pd.set_option('display.max_rows', 50)
        print(result_df)


In [ ]:
import akshare as ak
import pandas as pd

In [ ]:
# 此单元格之前因代理问题报错，已在上面的函数中统一处理代理清理
